In [ ]:
!pip install lightgbm

# Идея: логарифмирование признаков

Почему отказались: подход не давал особой разницы с моделью на сырых данных. Не решалась проблема выбросов на SI

Попробуем логарифмирование признаков, а не таргетов (Feature Log Transformation)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import KFold
from sklearn.metrics import root_mean_squared_error, r2_score
from xgboost import XGBRegressor
import warnings
warnings.filterwarnings('ignore')
from catboost import CatBoostRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_regression
from lightgbm import LGBMRegressor
from sklearn.cluster import KMeans

In [2]:
# Загружаем данные
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

train.columns = train.columns.str.strip()
test.columns = test.columns.str.strip()

In [3]:
IC50_PHYSICAL_MIN = train['IC50, mM'].min() # 0.003517

In [4]:
target_cols_all = ['IC50, mM', 'CC50, mM', 'SI']
feature_cols = [col for col in test.columns if col not in target_cols_all and col != 'index']

# Заполняем пропуски в сырых данных для корректной работы K-Means
X_train_raw = train[feature_cols].fillna(train[feature_cols].median())
X_test_raw = test[feature_cols].fillna(train[feature_cols].median())


In [5]:
# === ШАГ 1: ЛОГАРИФМИРОВАНИЕ ПРИЗНАКОВ (FEATURE LOG TRANSFORMATION) ===
# Защищаем от отрицательных значений перед логарифмом
for col in feature_cols:
    min_val = min(X_train_raw[col].min(), X_test_raw[col].min())
    if min_val < 0:
        X_train_raw[col] = X_train_raw[col] - min_val
        X_test_raw[col] = X_test_raw[col] - min_val
        
X_train_raw = np.log1p(X_train_raw)
X_test_raw = np.log1p(X_test_raw)

In [6]:
# === ШАГ 2: КЛАСТЕРИЗАЦИЯ НА ОДНОРОДНЫХ ПРИЗНАКАХ ===
X_all = pd.concat([X_train_raw, X_test_raw], axis=0)
kmeans = KMeans(n_clusters=12, random_state=42) # Увеличим число кластеров до 12
all_clusters = kmeans.fit_predict(X_all)

train['cluster'] = all_clusters[:len(train)]
test['cluster'] = all_clusters[len(train):]

cluster_means = train.groupby('cluster')[['IC50, mM', 'CC50, mM']].mean().reset_index()
cluster_means.columns = ['cluster', 'cluster_mean_IC50', 'cluster_mean_CC50']

X_train_raw = X_train_raw.merge(train[['cluster']], left_index=True, right_index=True)
X_train_raw = X_train_raw.merge(cluster_means, on='cluster', how='left').drop(columns=['cluster'])

X_test_raw = X_test_raw.merge(test[['cluster']], left_index=True, right_index=True)
X_test_raw = X_test_raw.merge(cluster_means, on='cluster', how='left').drop(columns=['cluster'])

submission_final = pd.DataFrame({'index': test['index']})
kf = KFold(n_splits=5, shuffle=True, random_state=42)

pure_test_preds = {}

In [7]:
for target in target_cols_all:
    y_train = train[target]
    
    # Отбор топ-110 лучших сглаженных признаков
    selector = SelectKBest(score_func=f_regression, k=110)
    X_train_selected = pd.DataFrame(selector.fit_transform(X_train_raw, y_train))
    X_test_selected = pd.DataFrame(selector.transform(X_test_raw))
    
    test_preds = np.zeros(len(test))
    rmse_scores = []
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X_train_selected, y_train)):
        X_tr, y_tr = X_train_selected.iloc[train_idx], y_train.iloc[train_idx]
        X_va, y_va = X_train_selected.iloc[val_idx], y_train.iloc[val_idx]
        
        model = CatBoostRegressor(
            iterations=2000,       # Даем модели больше времени сойтись на сглаженных признаках
            learning_rate=0.012,
            depth=6,
            loss_function='RMSE',
            l2_leaf_reg=6,
            random_seed=42,
            verbose=False
        )
        
        model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], early_stopping_rounds=120, verbose=False)
        val_preds = model.predict(X_va)
        rmse_scores.append(root_mean_squared_error(y_va, val_preds))
        
        test_preds += model.predict(X_test_selected) / kf.n_splits
        
    print(f"CatBoost [{target}] -> Локальный CV RMSE: {np.mean(rmse_scores):.4f}")
    pure_test_preds[target] = test_preds


CatBoost [IC50, mM] -> Локальный CV RMSE: 314.6416
CatBoost [CC50, mM] -> Локальный CV RMSE: 434.2098
CatBoost [SI] -> Локальный CV RMSE: 592.9381


In [8]:
# Шаг 3: Сборка финального ответа
submission_final['IC50'] = np.clip(pure_test_preds['IC50, mM'], 0.0, None)
submission_final['CC50'] = np.clip(pure_test_preds['CC50, mM'], 0.0, None)

protected_ic50 = np.clip(submission_final['IC50'], IC50_PHYSICAL_MIN, None)
si_strategy_math = submission_final['CC50'] / protected_ic50
si_strategy_model = np.clip(pure_test_preds['SI'], 0.0, None)

submission_final['SI'] = 0.75 * si_strategy_math + 0.25 * si_strategy_model
submission_final['SI'] = np.clip(submission_final['SI'], 0.0, train['SI'].max())

print("\nПроверка средних значений нового ансамбля:")
print(submission_final[['IC50', 'CC50', 'SI']].mean())

submission_final.to_csv('log_features_submission.csv', index=False)
print("\n[ГОТОВО] Файл 'log_features_submission.csv' успешно создан.")


Проверка средних значений нового ансамбля:
IC50    242.899196
CC50    642.582061
SI       24.547309
dtype: float64

[ГОТОВО] Файл 'log_features_submission.csv' успешно создан.
